# 索引管理与自动对齐

学习目标：管理表格索引，按行列标签完成运算、赋值和筛选，判断缺失标签及位置转换对结果的影响。

前置知识：Series 与 DataFrame、标签选择、算术运算。

运行环境：Python 3.12、pandas 3.0、NumPy 2.5。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

示例使用单元内构造的小表，后续单元沿用已导入的 pd。

## 1 按标签合计

两批商品的数量顺序不同，仍应把同一商品的数量相加。Series 算术先匹配标签，再计算对应值，不按当前位置配对。

下面的 A、B、C 是商品编号，数值单位为件。先核对 A 的结果：第一批 10 件加第二批 2 件，得到 12 件。

In [1]:
import pandas as pd

first = pd.Series([10, 20, 30], index=["A", "B", "C"], name="units")
second = pd.Series([3, 2, 4], index=["C", "A", "B"], name="units")
total = first + second
print(total)  # A、B、C 分别为 12、24、33 件，dtype 为 int64。
print(total.index.tolist(), total.shape)  # ['A', 'B', 'C'] (3,)

A    12
B    24
C    33
Name: units, dtype: int64
['A', 'B', 'C'] (3,)


## 2 Index 与 RangeIndex

Index 保存行或列的标签，用于选择和对齐。它不能逐项原地改写；调整标签时使用 rename 等方法，或替换整个索引。

RangeIndex 是表示等步长整数范围的索引，常见于未指定标签的列表输入。整数标签仍是标签，不能因为它恰好等于位置，就混淆 loc 与 iloc。

In [2]:
labels = pd.Index(["A", "B", "C"], name="product")
table = pd.DataFrame({"units": [10, 20, 30]})
print(labels)  # 标签 A、B、C，索引名称为 product。
print(table.index)  # RangeIndex(start=0, stop=3, step=1)
print(pd.RangeIndex(10, 16, 2).tolist())  # [10, 12, 14]，不包含 stop=16。

Index(['A', 'B', 'C'], dtype='str', name='product')
RangeIndex(start=0, stop=3, step=1)
[10, 12, 14]


## 3 设置、恢复与重命名索引

商品编号原本是普通列时，set_index 可以把它设为行标签。默认从数据列中移除该列，记录顺序不变；需要同时保留普通列时可设置 drop=False。

In [3]:
raw = pd.DataFrame({"product": ["B", "A", "C"], "units": [20, 10, 30]})
indexed = raw.set_index("product")
print(indexed)  # 行标签顺序 B、A、C，唯一数据列为 units。
print(indexed.index.name, indexed.shape)  # product (3, 1)
print(raw.columns.tolist())  # ['product', 'units']：默认返回新对象。

         units
product       
B           20
A           10
C           30
product (3, 1)
['product', 'units']


沿用 indexed，reset_index 默认把原索引放回普通列，并生成从 0 开始的 RangeIndex。drop=True 会丢弃原索引值，只保留数据列；应先判断编号是否还需要保留。

In [4]:
restored = indexed.reset_index()
discarded = indexed.reset_index(drop=True)
print(restored)  # product 列保留 B、A、C；行标签变为 0、1、2。
print(restored.shape, discarded.shape)  # (3, 2) (3, 1)
print(discarded.columns.tolist())  # ['units']：原编号已不在结果中。

  product  units
0       B     20
1       A     10
2       C     30
(3, 2) (3, 1)
['units']


rename 改变标签的名称，不按新标签重排数据。index 指定行标签映射，columns 指定列名映射；未列出的标签保持原样。需要发现映射中的拼写错误时使用 errors="raise"。

In [5]:
table = pd.DataFrame({"units": [20, 10]}, index=["B", "A"])
renamed = table.rename(index={"B": "P-B"}, columns={"units": "quantity"})
print(renamed)  # P-B 对应 20，A 对应 10；列名变为 quantity，行顺序不变。
print(renamed.shape)  # (2, 1)
try:
    table.rename(columns={"unit": "quantity"}, errors="raise")
except KeyError:
    print("不存在 unit 列，重命名被拒绝")  # 捕获预期的 KeyError。
else:
    raise AssertionError("拼写错误不应被忽略")

     quantity
P-B        20
A          10
(2, 1)
不存在 unit 列，重命名被拒绝


## 4 唯一性与重新索引

### 4.1 检查标签唯一性

pandas 允许重复标签，set_index 也不会自动保证编号唯一。标签重复时，一次标签选择可能对应多行，重新索引还可能因无法确定对应关系而失败。

index.is_unique 和 columns.is_unique 分别检查行、列标签。下面保留重复输入以观察边界；真实任务应先按编号含义决定如何处理重复，不能直接删掉其中一条。

In [6]:
raw = pd.DataFrame({"product": ["A", "A", "B"], "units": [10, 12, 20]})
table = raw.set_index("product")
print(table.index.is_unique, table.columns.is_unique)  # False True
print(table.loc["A"])  # 两行，units 为 10、12；结果是 DataFrame。
try:
    table.reindex(["A", "B", "C"])
except ValueError:
    print("重复行标签不能完成这次 reindex")  # 捕获预期的 ValueError。
else:
    raise AssertionError("本例重复标签应导致重新索引失败")

False True
         units
product       
A           10
A           12
重复行标签不能完成这次 reindex


### 4.2 按目标标签排列

reindex 按给定的目标标签选取、重排或补出行列。原数据没有的标签默认产生缺失值；未列入目标的原标签不保留。它与 rename 的区别是：reindex 查找标签对应的数据，rename 改标签本身。

下面的目标行顺序为 C、A、D，D 没有原记录。普通 int64 列出现 NaN 后，本例会转换为 float64。

In [7]:
table = pd.DataFrame({"units": [10, 20, 30]}, index=["A", "B", "C"])
ordered = table.reindex(index=["C", "A", "D"], columns=["units", "reserved"])
print(ordered)  # units 为 30、10、NaN；新列 reserved 全部缺失。
print(ordered.index.tolist(), ordered.columns.tolist(), ordered.shape)
# ['C', 'A', 'D']、['units', 'reserved']，形状为 (3, 2)。
print(ordered.dtypes)  # 两列均为 float64。

   units  reserved
C   30.0       NaN
A   10.0       NaN
D    NaN       NaN
['C', 'A', 'D'] ['units', 'reserved'] (3, 2)
units       float64
reserved    float64
dtype: object


## 5 显式对齐两个对象

Series 算术默认保留双方标签的并集；只有一方提供数值的标签，结果通常缺失。若只需双方都有的记录，应先选择交集。

DataFrame.align 一次返回两个对齐后的对象。axis="index" 只处理行，axis="columns" 只处理列，axis=None 同时处理两轴。join 决定保留哪些标签。

| join 取值 | 中文名称／含义 |
| --- | --- |
| outer | 并集；默认方式，按标签排序 |
| inner | 交集，保留左侧标签的顺序 |
| left | 使用左侧标签及顺序 |
| right | 使用右侧标签及顺序 |

In [8]:
left = pd.DataFrame({"units": [20, 10], "reserved": [2, 1]}, index=["B", "A"])
right = pd.DataFrame({"units": [3, 4], "returned": [1, 0]}, index=["A", "C"])
left_inner, right_inner = left.align(right, join="inner")
print(left_inner, right_inner, sep="\n")  # 都只保留 A 行、units 列，值分别为 10、3。
print((left_inner + right_inner))  # A 的 units 为 13，形状为 (1, 1)。

left_outer, right_outer = left.align(right, join="outer")
print(left_outer.index.tolist(), left_outer.columns.tolist())
# 行为 A、B、C；列为 reserved、returned、units。
print(left_outer.shape, right_outer.shape)  # (3, 3) (3, 3)
print(left_outer.loc["C"].isna().all())  # True：左侧没有 C 的记录。

   units
A     10
   units
A      3
   units
A     13
['A', 'B', 'C'] ['reserved', 'returned', 'units']
(3, 3) (3, 3)
True


## 6 DataFrame 与 Series 的行列运算

### 6.1 指定匹配方向

同一张库存表有三个门店、两个时段，数值单位为件。基准可能“每个门店一个数”，也可能“每个时段一个数”。DataFrame 的二元运算用 axis 指定 Series 标签应匹配哪条轴。

（1）axis="index" 匹配表的行标签：某个门店的基准用于该行所有列。

（2）axis="columns" 匹配表的列标签：某个时段的基准用于该列所有行。这也是默认方向。

| 方法 | 中文名称／含义 | 对应运算 |
| --- | --- | --- |
| add | 加法 | 表格值加匹配值 |
| sub | 减法 | 表格值减匹配值 |
| mul | 乘法 | 表格值乘匹配值 |
| div | 除法 | 表格值除以匹配值 |

先用 sub 比较两类基准。两个 Series 都故意打乱标签顺序。对齐可能改变结果的列顺序；下面在按列运算后用 reindex 恢复原表的列顺序，便于逐项比较。

In [9]:
stock = pd.DataFrame({"morning": [12, 18, 24], "evening": [20, 30, 40]}, index=["A", "B", "C"])
row_base = pd.Series([4, 2, 3], index=["C", "A", "B"])
column_base = pd.Series([20, 10], index=["evening", "morning"])

by_row = stock.sub(row_base, axis="index")
by_column = stock.sub(column_base, axis="columns")
print(by_column.columns.tolist())  # ['evening', 'morning']：先观察实际对齐后的列序。
by_column = by_column.reindex(columns=stock.columns)
print(by_row)  # A：[10, 18]；B：[15, 27]；C：[20, 36]。
print(by_column)  # A：[2, 0]；B：[8, 10]；C：[14, 20]。
print(by_row.shape, by_column.shape)  # 都为 (3, 2)，行列标签与 stock 一致。

['evening', 'morning']


   morning  evening
A       10       18
B       15       27
C       20       36
   morning  evening
A        2        0
B        8       10
C       14       20
(3, 2) (3, 2)


沿用 stock 和两个基准 Series。省略 axis 时仍按列匹配；不会因为 Series 长度等于行数就自动改成按行匹配。行标签 A、B、C 若被当作列名，会与原列取并集，产生额外列和缺失值。

In [10]:
default_columns = stock.sub(column_base)
wrong_axis = stock.sub(row_base)
print(default_columns.equals(stock - column_base))  # True：运算符同样默认匹配列。
print(default_columns.reindex(columns=stock.columns).equals(by_column))  # True：列序统一后相同。
print(wrong_axis.columns.tolist(), wrong_axis.shape)
# ['A', 'B', 'C', 'evening', 'morning']，形状为 (3, 5)。
print(wrong_axis.isna().all().all())  # True：两侧没有可匹配的列标签。

True


True
['A', 'B', 'C', 'evening', 'morning'] (3, 5)
True


### 6.2 加法、乘法与除法

沿用 stock。每个门店追加同样数量到两个时段时用 axis="index"；按时段分别追加时用 axis="columns"。add 与 sub 使用相同的标签匹配规则。

In [11]:
row_extra = pd.Series([4, 2, 3], index=["C", "A", "B"])
column_extra = pd.Series([2, 1], index=["evening", "morning"])
print(stock.add(row_extra, axis="index"))  # A：[14, 22]；B：[21, 33]；C：[28, 44]。
print(stock.add(column_extra, axis="columns"))  # evening 为 22、32、42；morning 为 13、19、25。

   morning  evening
A       14       22
B       21       33
C       28       44
   evening  morning
A       22       13
B       32       19
C       42       25


mul 和 div 也采用相同方向约定。沿用 stock，用无量纲系数分别按门店或时段放大、缩小；下面的除数均非零。

In [12]:
row_factor = pd.Series([4, 2, 3], index=["C", "A", "B"])
column_factor = pd.Series([10, 2], index=["evening", "morning"])
print(stock.mul(row_factor, axis="index"))  # A：[24, 40]；B：[54, 90]；C：[96, 160]。
print(stock.mul(column_factor, axis="columns"))  # morning：[24, 36, 48]；evening：[200, 300, 400]。
print(stock.div(row_factor, axis="index"))  # 每行均为 [6.0, 10.0]。
print(stock.div(column_factor, axis="columns"))  # morning：[6.0, 9.0, 12.0]；evening：[2.0, 3.0, 4.0]。

   morning  evening
A       24       40
B       54       90
C       96      160
   evening  morning
A      200       24
B      300       36
C      400       48
   morning  evening
A      6.0     10.0
B      6.0     10.0
C      6.0     10.0
   evening  morning
A      2.0      6.0
B      3.0      9.0
C      4.0     12.0


## 7 两个 DataFrame 的缺失填充

两个 DataFrame 运算时同时对齐行与列。add 的 fill_value 可在计算前填充单侧缺失，包括原有缺失以及标签不匹配产生的缺失；双方同一位置都缺失时，结果仍然缺失。

下面单独用两张表说明 fill_value=0：只有一方有记录时采用已有数值，双方都没有数值时保留未知。把缺失当作零必须符合任务含义；此参数不能理解为把最终所有缺失结果都改成零。本节的 fill_value 示例仅针对两个 DataFrame。

In [13]:
left = pd.DataFrame(
    {"morning": [10.0, float("nan"), float("nan")], "evening": [20.0, float("nan"), 6.0]},
    index=["A", "B", "C"],
)
right = pd.DataFrame(
    {"morning": [4.0, 3.0, 8.0], "evening": [float("nan"), 2.0, 5.0]},
    index=["C", "A", "D"],
)
ordinary = left + right
filled = left.add(right, fill_value=0)
print(ordinary)  # 只有 A 得到 [13.0, 22.0]，B、C、D 的两列均缺失。
print(filled)  # A：[13, 22]；B：双侧缺失；C：[4, 6]；D：[8, 5]。
print(filled.index.tolist(), filled.shape)  # ['A', 'B', 'C', 'D'] (4, 2)
print(filled.loc["B"].isna().all())  # True：fill_value 没有覆盖双侧缺失。

   morning  evening
A     13.0     22.0
B      NaN      NaN
C      NaN      NaN
D      NaN      NaN


   morning  evening
A     13.0     22.0
B      NaN      NaN
C      4.0      6.0
D      8.0      5.0
['A', 'B', 'C', 'D'] (4, 2)
True


## 8 赋值时的标签对齐

把 Series 赋给一列时，值按表的行标签放回。Series 的额外标签不会扩展已有表的行；表中有、Series 中没有的标签会得到缺失值。

下面的预留数量按 C、A、D 排列，赋值后仍保留表的 A、B、C 行顺序。

In [14]:
table = pd.DataFrame({"units": [10, 20, 30]}, index=["A", "B", "C"])
reserved = pd.Series([3, 1, 9], index=["C", "A", "D"])
table["reserved"] = reserved
print(table)  # reserved 在 A、B、C 行分别为 1.0、NaN、3.0；D 不加入表格。
print(table.shape, table["reserved"].dtype)  # (3, 2) float64

   units  reserved
A     10       1.0
B     20       NaN
C     30       3.0
(3, 2) float64


loc 接收 DataFrame 作为赋值右侧时，会匹配行列两条轴。即使右侧把行列同时倒序，值仍按自己的标签赋给目标位置。

In [15]:
target = pd.DataFrame({"morning": [0, 0], "evening": [0, 0]}, index=["A", "B"])
incoming = pd.DataFrame({"evening": [40, 20], "morning": [30, 10]}, index=["B", "A"])
target.loc[:, ["morning", "evening"]] = incoming
print(target)  # A：[10, 20]；B：[30, 40]，列顺序仍为 morning、evening。
print(target.index.tolist(), target.shape)  # ['A', 'B'] (2, 2)

   morning  evening
A       10       20
B       30       40
['A', 'B'] (2, 2)


## 9 布尔标签与位置语义

布尔 Series 用于 loc 筛选时，先把掩码标签对齐到表的行标签。下面所有标签均唯一，掩码覆盖全部目标行；筛选结果保持表中被选行的顺序。

to_numpy 去掉 Series 标签。此时布尔数组只按当前位置对应行，长度相同并不能证明对应关系正确。iloc 接受布尔数组；本例带字符串标签的布尔 Series 会被拒绝。

官方用户指南将布尔 Series 列为 iloc 不支持的输入。虽然 pandas 3.0.6 实现会放行部分整数索引的布尔 Series 并按标签对齐，这与文档约定不一致，不能据此推断它按位置筛选。表达标签条件用 loc；表达位置条件用长度一致且顺序已确认的布尔数组。

In [16]:
table = pd.DataFrame({"units": [10, 20, 30]}, index=["A", "B", "C"])
keep = pd.Series([True, False, True], index=["C", "A", "B"])
by_label = table.loc[keep]
by_position = table.iloc[keep.to_numpy()]
print(by_label)  # B、C 行：20、30。
print(by_position)  # A、C 行：10、30；标签已丢失，直接使用掩码的当前位置。
print(by_label.shape, by_position.shape)  # 都为 (2, 1)，但选出的记录不同。

   units
B     20
C     30
   units
A     10
C     30
(2, 1) (2, 1)


掩码缺少目标行标签时无法直接对齐，会抛出 IndexingError。若任务明确规定“没有筛选记录的行不保留”，可先按表索引 reindex 并把新增位置填为 False；若没有这个规定，就应先补齐或核查掩码。

In [17]:
table = pd.DataFrame({"units": [10, 20, 30]}, index=["A", "B", "C"])
partial = pd.Series([True, False], index=["A", "C"])
try:
    table.loc[partial]
except pd.errors.IndexingError:
    print("掩码缺少 B 标签，无法直接对齐")  # 捕获预期的 IndexingError。
else:
    raise AssertionError("不完整的掩码不应直接通过")

complete = partial.reindex(table.index, fill_value=False)
print(complete)  # A=True，B=False，C=False，dtype 为 bool。
print(table.loc[complete])  # 只保留 A 行，units 为 10。

掩码缺少 B 标签，无法直接对齐
A     True
B    False
C    False
dtype: bool
   units
A     10


## 10 to_numpy 的类型与顺序

只有任务明确按位置对应时，才应去掉标签。若希望按标签对应但下游只接收数组，应先 reindex 到目标顺序，再转换。

下面对照直接按位置赋值，以及先对齐再转换的结果。数组长度必须与目标行数一致。

In [18]:
table = pd.DataFrame({"units": [10, 20, 30]}, index=["A", "B", "C"])
reserved = pd.Series([3, 1, 2], index=["C", "A", "B"])
table["position"] = reserved.to_numpy()
table["aligned"] = reserved.reindex(table.index).to_numpy()
print(table)  # position 为 3、1、2；aligned 为 1、2、3。
print(table.shape)  # (3, 3)，行标签仍为 A、B、C。

   units  position  aligned
A     10         3        1
B     20         1        2
C     30         2        3
(3, 3)


DataFrame 各列可以有不同 dtype，但普通二维 NumPy 数组只有一个 dtype。to_numpy 会选择共同类型；整数与浮点数混合时可能统一为浮点，数值与字符串混合时可能得到 object。

转换也可能复制或强制转换值，默认 copy=False 不保证零复制。检查标签之外，还要检查转换后的形状、类型及列顺序。

In [19]:
numeric = pd.DataFrame({"units": [1, 2], "ratio": [0.5, 1.5]}, index=["A", "B"])
values = numeric.to_numpy()
print(numeric.dtypes)  # units 为 int64，ratio 为 float64。
print(values, values.shape, values.dtype)  # [[1.0, 0.5], [2.0, 1.5]] (2, 2) float64

mixed = pd.DataFrame({"code": ["A", "B"], "units": [1, 2]})
print(mixed.to_numpy().dtype)  # object：不能保留各列独立 dtype。

units      int64
ratio    float64
dtype: object
[[1.  0.5]
 [2.  1.5]] (2, 2) float64
object


## 11 选学：索引的集合关系

对齐前可先查看双方共有、全部涉及和单方独有的标签。以下输入标签唯一；集合操作用于比较标签，不是统计重复记录的次数。显式设置 sort=False，避免额外排序。

| 方法 | 中文名称／含义 |
| --- | --- |
| union | 并集，双方出现的标签 |
| intersection | 交集，双方都有的标签 |
| difference | 差集，左侧有而右侧没有的标签 |

In [20]:
left = pd.Index(["B", "A", "C"])
right = pd.Index(["C", "D", "A"])
print(left.union(right, sort=False).tolist())  # ['B', 'A', 'C', 'D']
print(left.intersection(right, sort=False).tolist())  # ['A', 'C']
print(left.difference(right, sort=False).tolist())  # ['B']
print(right.difference(left, sort=False).tolist())  # ['D']：差集有方向。

['B', 'A', 'C', 'D']
['A', 'C']
['B']
['D']


## 本章小结

（1）set_index 和 reset_index 在普通列与行标签之间转换；rename 改标签，reindex 按目标标签查找和排列数据。

（2）算术、Series 列赋值和布尔 Series 筛选都涉及标签对齐，但保留标签的规则不同；应核对唯一性、顺序、形状与缺失位置。

（3）DataFrame 与 Series 运算默认匹配列标签。axis="index" 匹配行，axis="columns" 匹配列，add、sub、mul、div 都遵循此规则。

（4）两个 DataFrame 运算的 fill_value 可处理单侧缺失，双侧缺失仍保留。to_numpy 则丢弃标签，并可能改变 dtype；位置对应必须另有明确约定。

## 练习

（1）将 product 设为索引并检查唯一性，按 C、A、D 重新排列，最后把编号恢复为普通列。说明 rename 为什么不能替代本题的 reindex。

In [21]:
raw = pd.DataFrame({"product": ["B", "A", "C"], "units": [20, 10, 30]})
# 在此完成转换并用 print() 检查。
# 检查：最终编号列为 C、A、D，units 为 30、10、NaN，shape 为 (3, 2)。
# 在注释中解释重新查找标签与改名的区别。

（2）先预测两个筛选结果的行标签，再运行核对。随后把要求改为“下游只能接收布尔数组，但仍须按编号筛选”，写出先对齐再转换的做法，并解释为什么直接 to_numpy 不满足要求。

In [22]:
table = pd.DataFrame({"units": [10, 20, 30]}, index=["A", "B", "C"])
keep = pd.Series([False, True, True], index=["C", "A", "B"])
# 先在此记录预测，不要只比较结果行数。
print(table.loc[keep].index.tolist())
print(table.iloc[keep.to_numpy()].index.tolist())
# 在此满足新的数组输入要求；检查：与按标签筛选选出同一批记录。

['A', 'B']
['B', 'C']


（3）对同一张表分别减去每个门店的基准、每个时段的基准。两次都保留原行列标签及顺序，解释各自选择 axis 的理由。随后按门店系数做除法，核对两个时段都使用了对应门店的系数。

In [23]:
stock = pd.DataFrame({"morning": [12, 18, 24], "evening": [20, 30, 40]}, index=["A", "B", "C"])
row_base = pd.Series([3, 4, 2], index=["B", "C", "A"])
column_base = pd.Series([20, 10], index=["evening", "morning"])
# 在此完成两次减法；检查 A 行分别为 [10, 18] 和 [2, 0]。
# 提示：按列对齐后检查列顺序，必要时使用 reindex。
# 用 row_base 作为除数；检查每行结果都为 [6.0, 10.0]，shape 为 (3, 2)。
# 在注释中解释匹配的是 Series 的哪些标签。

（4）合计两个 DataFrame：单侧缺失按零处理，双方同位置缺失仍视为未知。输出结果并检查缺失位置。若业务进一步要求“原始缺失必须先核查，不能按零处理”，应改变哪一步？说明理由。

In [24]:
left = pd.DataFrame({"units": [10.0, float("nan"), float("nan")]}, index=["A", "B", "C"])
right = pd.DataFrame({"units": [3.0, float("nan"), 5.0]}, index=["C", "B", "A"])
# 在此按最初规则合计；检查 A=15、B 缺失、C=3，标签为 A、B、C。
# 在注释中说明新约束下是否仍应使用 fill_value=0。

## 参考与引用来源

在线文档可能随发布更新；固定版本对照见下表 pandas v3.0.6 文档源码。API 参数与异常仍须结合所列页面的具体定位阅读。

| 网站 | 本章参考内容与定位 |
| --- | --- |
| pandas 官方在线文档（课程基线 3.0.6） | [Index](https://pandas.pydata.org/docs/reference/api/pandas.Index.html) 的不可变轴标签；[RangeIndex](https://pandas.pydata.org/docs/reference/api/pandas.RangeIndex.html) 的整数范围与默认索引；[set_index](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.set_index.html) 的 drop、默认行为；[reset_index](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.reset_index.html) 的 drop 与原标签恢复；[rename](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rename.html) 的行列映射与 errors；[reindex](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.reindex.html) 的目标标签和缺失值；[Duplicate Labels](https://pandas.pydata.org/docs/user_guide/duplicates.html) 的 Consequences of Duplicate Labels、Duplicate Label Detection；[align](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.align.html) 的 join、axis 与返回值；[add](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.add.html)、[sub](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sub.html)、[mul](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.mul.html)、[div](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.div.html) 的 Series 匹配轴与 DataFrame 的 fill_value；[Intro to data structures](https://pandas.pydata.org/docs/user_guide/dsintro.html) 的 Vectorized operations and label alignment with Series、Column selection, addition, deletion、Data alignment and arithmetic；[loc](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.loc.html) 的布尔 Series 对齐、IndexingError 和赋值示例；[Indexing and selecting data](https://pandas.pydata.org/docs/user_guide/indexing.html) 的 Basics、Boolean indexing 与标签赋值；[DataFrame.to_numpy](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_numpy.html)、[Series.to_numpy](https://pandas.pydata.org/docs/reference/api/pandas.Series.to_numpy.html) 的值转换、dtype 与 copy；[union](https://pandas.pydata.org/docs/reference/api/pandas.Index.union.html)、[intersection](https://pandas.pydata.org/docs/reference/api/pandas.Index.intersection.html)、[difference](https://pandas.pydata.org/docs/reference/api/pandas.Index.difference.html) 的集合关系与 sort。 |
| GitHub 官方项目（版本化来源） | pandas v3.0.6 文档源码：[duplicates](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/user_guide/duplicates.rst)、[dsintro](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/user_guide/dsintro.rst)、[indexing](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/user_guide/indexing.rst)；对应上列同名指南或发布说明的小节，作为固定版本对照。 [indexing.py](https://github.com/pandas-dev/pandas/blob/v3.0.6/pandas/core/indexing.py#L1592-L1600) 的 _iLocIndexer._validate_key、[第 1222 行](https://github.com/pandas-dev/pandas/blob/v3.0.6/pandas/core/indexing.py#L1222-L1227) 的 _getbool_axis 与 [check_bool_indexer](https://github.com/pandas-dev/pandas/blob/v3.0.6/pandas/core/indexing.py#L2647)：整数索引布尔 Series 的放行及标签对齐，与指南 Boolean indexing 的限制不一致，仅说明本版本实现。 |